# Telecom Churn Prediction: Phase 5 to 9 - Model Training & Evaluation

This notebook covers:
1. Stratified Train/Test Split (Preventing Data Leakage)
2. Building sklearn Preprocessing Pipeline (`ColumnTransformer`, `StandardScaler`, `OneHotEncoder`)
3. Logistic Regression Training, 5-Fold Cross-Validation & Hyperparameter Tuning (`GridSearchCV`)
4. Model Evaluation on Locked Test Set (ROC-AUC, PR-AUC, Confusion Matrix, Calibration Curve)
5. Feature Importance / Coefficient Analysis

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('..'))
from src.model_trainer import ModelTrainer
from src.pipeline import build_preprocessing_pipeline
from src.evaluator import ModelEvaluator

## 1. Load Processed Data & Split
We load the clean processed data and apply a stratified 80/20 train/test split. `X_test` and `y_test` remain locked away.

In [ ]:
processed_path = os.path.join('..', 'data', 'processed', 'processed_churn.csv')
df_clean = pd.read_csv(processed_path)

trainer = ModelTrainer(target_col='Churn', id_col='customerID')
X_train, X_test, y_train, y_test, ids_train, ids_test = trainer.split_data(df_clean)
print(f"Train shape: {X_train.shape} | Test shape: {X_test.shape}")

## 2. Preprocessing & Model Pipeline

In [ ]:
preprocessor, num_cols, cat_cols = build_preprocessing_pipeline(df_clean, target_col='Churn', id_col='customerID')
model = trainer.train_logistic_regression(X_train, y_train, preprocessor, tune_c=True)

## 3. Evaluation on Test Set

In [ ]:
plots_dir = os.path.join('..', 'outputs', 'plots')
evaluator = ModelEvaluator(output_dir=plots_dir)
churn_probs, roc_auc, pr_auc = evaluator.evaluate_model(model, X_test, y_test, model_name='Logistic Regression')
evaluator.plot_roc_pr_curves(y_test, churn_probs, model_name='Logistic Regression')
evaluator.plot_calibration_curve(y_test, churn_probs, model_name='Logistic Regression')
coef_df = evaluator.extract_and_plot_coefficients(model)